In [2]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 556.4 kB/s eta 0:02:09
   ---------------------------------------- 0.5/72.0 MB 556.4 kB/s eta 0:02:09
   ---------------------------------------- 0.5/72.0 MB 556.4 kB/s eta 0:02:09
   ---------------------------------------- 0.5/72.0 MB 556.4 kB/s eta 0:02:09
   ---------------------------------------- 0.8/72.0 MB 408.2 kB/s eta 0:02:55
   ---------------------------------------- 0.8/72.0 MB 408.2 kB/s eta 0:02:55
   ---------------------------------------- 0.8/72.0 MB 408.2 kB/s eta 0:02:55
   -----------------------------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pickle
import pandas as pd
import numpy as np

# =========================================================
# 1. Load the trained model and preprocessing artifacts
# =========================================================
with open("best_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("preprocessing_artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

# =========================================================
# 2. Preprocess a new movie (revised & stable)
# =========================================================
def preprocess_new_movie(movie_data, artifacts):
    """
    Clean and align new movie data for prediction.
    Uses the same encodings and transformations as training.
    """

    def parse_list(text):
        if pd.isna(text) or text == "":
            return []
        return [t.strip() for t in str(text).split(",")]

    f = {}  # features dict

    # ---------- ACTORS ----------
    actors = parse_list(movie_data.get("actors", ""))
    actor_ratings = [
        artifacts["actor_avg_ratings"].get(a, artifacts["global_avg_rating"])
        for a in actors
    ]
    f["actors_rating_mean"] = np.mean(actor_ratings) if actor_ratings else artifacts["global_avg_rating"]
    f["actors_rating_max"] = np.max(actor_ratings) if actor_ratings else artifacts["global_avg_rating"]
    f["actors_rating_min"] = np.min(actor_ratings) if actor_ratings else artifacts["global_avg_rating"]
    f["num_actors"] = len(actors)
    f["num_top_actors"] = sum(1 for a in actors if a in artifacts["top_actors"])

    for a in artifacts["top_actors"]:
        f[f"actor_{a}"] = 1 if a in actors else 0

    # ---------- DIRECTOR ----------
    directors = parse_list(movie_data.get("director", ""))
    director_ratings = [
        artifacts["director_avg_ratings"].get(d, artifacts["global_avg_rating"])
        for d in directors
    ]
    f["director_rating_mean"] = np.mean(director_ratings) if director_ratings else artifacts["global_avg_rating"]
    f["director_rating_max"] = np.max(director_ratings) if director_ratings else artifacts["global_avg_rating"]
    f["director_rating_min"] = np.min(director_ratings) if director_ratings else artifacts["global_avg_rating"]
    f["has_top_director"] = int(any(d in artifacts["top_directors"] for d in directors))

    for d in artifacts["top_directors"]:
        f[f"director_{d}"] = 1 if d in directors else 0

    # ---------- GENRES ----------
    genres = parse_list(movie_data.get("genres", ""))
    f["num_genres"] = len(genres)
    for g in artifacts["all_genres"]:
        f[g] = 1 if g in genres else 0

    # ---------- DATE ----------
    month = movie_data.get("release_month", 6)
    f["release_month"] = month
    f["release_day_of_week"] = movie_data.get("release_day_of_week", 0)
    f["release_quarter"] = (month - 1) // 3 + 1
    f["is_summer_release"] = int(month in [5, 6, 7, 8])
    f["is_holiday_release"] = int(month in [11, 12])

    # ---------- NUMERIC FEATURES (safe-scaled) ----------
    budget = movie_data.get("budget", 0)
    revenue = movie_data.get("revenue", 0)
    runtime = movie_data.get("runtime", 120)
    popularity = movie_data.get("popularity", 10)
    vote_count = movie_data.get("vote_count", 100)

    # keep only log-transformed versions (cleaner)
    f["budget_log"] = np.log1p(budget)
    f["revenue_log"] = np.log1p(revenue)
    profit = revenue - budget
    f["profit_log"] = np.log1p(max(0, profit))
    f["roi"] = (revenue - budget) / budget if budget > 0 else 0
    f["popularity_log"] = np.log1p(popularity)
    f["vote_count_log"] = np.log1p(vote_count)
    f["budget_per_minute"] = budget / runtime if runtime > 0 else 0
    f["popularity_per_vote"] = popularity / max(1, vote_count)
    f["is_standard_length"] = int(90 <= runtime <= 130)

    # categorical buckets (kept consistent)
    for cat in ["Zero", "Low", "Medium", "High", "Blockbuster"]:
        f[f"budget_cat_{cat}"] = 0
    if budget == 0:
        f["budget_cat_Zero"] = 1
    elif budget <= 1e6:
        f["budget_cat_Low"] = 1
    elif budget <= 10e6:
        f["budget_cat_Medium"] = 1
    elif budget <= 50e6:
        f["budget_cat_High"] = 1
    else:
        f["budget_cat_Blockbuster"] = 1

    # runtime categories
    for cat in ["Short", "Medium", "Long", "VeryLong"]:
        f[f"runtime_{cat}"] = 0
    if runtime <= 90:
        f["runtime_Short"] = 1
    elif runtime <= 120:
        f["runtime_Medium"] = 1
    elif runtime <= 150:
        f["runtime_Long"] = 1
    else:
        f["runtime_VeryLong"] = 1

    # popularity categories
    for cat in ["Low", "Medium", "High", "VeryHigh"]:
        f[f"pop_{cat}"] = 0
    if popularity <= 10:
        f["pop_Low"] = 1
    elif popularity <= 50:
        f["pop_Medium"] = 1
    elif popularity <= 100:
        f["pop_High"] = 1
    else:
        f["pop_VeryHigh"] = 1

    # ---------- LANGUAGE ----------
    lang = movie_data.get("original_language", "en")
    for col in artifacts["feature_names"]:
        if col.startswith("lang_"):
            f[col] = 1 if lang == col.replace("lang_", "") else 0

    # ---------- OTHER ----------
    f["year"] = movie_data.get("year", 2025)
    overview = str(movie_data.get("overview", ""))
    f["overview_length"] = len(overview)
    f["overview_word_count"] = len(overview.split())

    # ---------- ALIGN ----------
    df = pd.DataFrame([f])
    for col in artifacts["feature_names"]:
        if col not in df.columns:
            df[col] = 0
    df = df[artifacts["feature_names"]].astype("float32")

    return df.values


# =========================================================
# 3. Prediction helper
# =========================================================
def predict_movie(movie_dict, artifacts, model, name):
    X = preprocess_new_movie(movie_dict, artifacts)
    y_pred = float(model.predict(X)[0])
    y_pred = np.clip(y_pred, 1.0, 10.0)

    print("\n" + "=" * 70)
    print(f"MOVIE: {name}")
    print("=" * 70)
    print(f"Predicted Rating: {y_pred:.2f} / 10")

    if y_pred >= 8:
        print("Assessment: 🌟 Excellent - Potential Award Contender")
    elif y_pred >= 7:
        print("Assessment: 👍 Good - Strong Audience Appeal")
    elif y_pred >= 6:
        print("Assessment: 😐 Average - Mixed Reception")
    else:
        print("Assessment: 👎 Below Average - Limited Appeal")
    return y_pred


# =========================================================
# 4. Define your two movies
# =========================================================
blockbuster = {
    "title": "Inception 2: The Awakening",
    'actors': 'Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page',
    'director': 'Christopher Nolan',
    'genres': 'Action, Sci-Fi, Thriller',
    'budget': 200_000_000,
    'revenue': 900_000_000,
    'runtime': 148,
    'popularity': 150,
    'vote_count': 30000,
    'year': 2025,
    'original_language': 'en',
    'release_month': 7,
    'overview': 'A sequel to the mind-bending dream heist film, delving deeper into layered realities.'
}

indie_film = {
    "title": "When the Dirt Cries",
    "actors": "Nabih Berry, Waleed Joumblat, Samir Geagea",
    "director": "Michel Aoun",
    "genres": "Political",
    "budget": 100_000,
    "revenue": 20_000,
    "runtime": 60,
    "popularity": 5.0,
    "vote_count": 50,
    "year": 2025,
    "original_language": "ar",
    "release_month": 3,
    "release_day_of_week": 1,
    "overview": "When Lebanon’s top political rivals team up to save their seats, chaos, comedy, and endless negotiations take over the nation."
}

predict_movie(blockbuster, artifacts, model, blockbuster["title"])
predict_movie(indie_film, artifacts, model, indie_film["title"])


MOVIE: Inception 2: The Awakening
Predicted Rating: 7.29 / 10
Assessment: 👍 Good - Strong Audience Appeal

MOVIE: When the Dirt Cries
Predicted Rating: 6.66 / 10
Assessment: 😐 Average - Mixed Reception


np.float64(6.662783145904541)